In [0]:
%python
from datetime import datetime, timedelta

In [0]:
%python
import yfinance as yf
import pandas as pd

In [0]:
%python
tickers = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

In [0]:
%python
end_date = datetime.today()
start_date = end_date - timedelta(days=30)

df_list = []

In [0]:
%python
df_list = []

for ticker in tickers:
    # O segredo: baixamos um por um e resetamos o índice imediatamente
    df = yf.download(ticker, start=start_date, end=end_date)
    df = df.reset_index()
    
    # Esta linha garante que, se vier (Close, PETR4), vire apenas "Close"
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    
    # Agora limpamos espaços e pontos para o Delta aceitar
    df.columns = [str(c).lower().replace(" ", "_").replace(".", "_") for c in df.columns]
    
    df["ticker"] = ticker
    df_list.append(df)

In [0]:
%python
# 3. Consolidação
pandas_df = pd.concat(df_list)

In [0]:
%python
# 4. Transformação para Spark (O motor de processamento em escala)
spark_df = spark.createDataFrame(pandas_df)

In [0]:
%python
# 5. Persistência na Camada Bronze (Delta Lake)
nome_tabela = "bronze_investimentos_alpha"

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(nome_tabela)

print(f"Sucesso! Tabela {nome_tabela} criada e salva no catálogo.")
display(spark_df)